In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

sem_eval_2026_task_13_subtask_a_path = kagglehub.competition_download('sem-eval-2026-task-13-subtask-a')

print('Data source import complete.')

100%|██████████| 448M/448M [00:17<00:00, 27.6MB/s]

Extracting files...


Data source import complete.


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [4]:
!pip install -q transformers accelerate datasets evaluate scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [5]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from dataclasses import dataclass
from typing import Any, Dict, List

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
import evaluate
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt
import gc

In [6]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {'GPU - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Using device: GPU - Tesla T4


In [7]:
# Mount Google Drive (for checkpoint saving)
from google.colab import drive
drive.mount("/content/drive")
CHECKPOINT_DIR = "/content/drive/MyDrive/semeval_taskA"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/semeval_taskA


In [8]:
train_df = pd.read_parquet('/root/.cache/kagglehub/competitions/sem-eval-2026-task-13-subtask-a/Task_A/train.parquet')
val_df = pd.read_parquet('/root/.cache/kagglehub/competitions/sem-eval-2026-task-13-subtask-a/Task_A/validation.parquet')
test_df = pd.read_parquet('/root/.cache/kagglehub/competitions/sem-eval-2026-task-13-subtask-a/Task_A/test.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

assert "code" in train_df.columns and "label" in train_df.columns
assert "code" in val_df.columns   and "label" in val_df.columns
assert "code" in test_df.columns  and "ID"    in test_df.columns

print(f"\nUsing full training set: {len(train_df)} samples")
print(f"Label distribution:\n{train_df['label'].value_counts()}")


Train shape: (500000, 4)
Validation shape: (100000, 4)
Test shape: (500000, 2)

Using full training set: 500000 samples
Label distribution:
label
1    261525
0    238475
Name: count, dtype: int64


In [9]:
_code_pattern = re.compile(
    r'[{}\[\]();]|==|!=|<=|>=|->|::|=>|'
    r'\b(if|else|for|while|return|def|class|'
    r'function|func|import|include|var|let|const|int|void)\b'
)
_text_pattern = re.compile(
    r'^#+\s|^//\s*[A-Z]|^/\*|^\s*\*|'
    r'[.!?]\s+[A-Z]|'
    r'\b(the|this|that|you|can|will|should|must|note|example)\b',
    re.IGNORECASE
)

def classify_and_prefix(code: str) -> str:
    if not code or len(code.strip()) < 10:
        return f"[TYPE:fragment LEN:short CLASS:0 LOOP:0 FUNC:0] {code}"

    lines = [l for l in code.strip().splitlines() if l.strip()]
    n     = max(len(lines), 1)

    code_lines = sum(1 for l in lines if _code_pattern.search(l))
    text_lines = sum(1 for l in lines if _text_pattern.search(l))
    code_ratio = code_lines / n
    text_ratio = text_lines / n

    if code_ratio >= 0.5:
        content_type = "clean"      # mostly real code
    elif text_ratio >= 0.4 or code_ratio < 0.2:
        content_type = "fragment"   # mostly text / pseudocode
    else:
        content_type = "mixed"      # code + natural language

    total_lines = len(code.strip().splitlines())
    complexity  = "short" if total_lines < 20 else ("medium" if total_lines < 60 else "long")
    has_class   = int(bool(re.search(r'\bclass\b', code)))
    has_loop    = int(bool(re.search(r'\bfor\b|\bwhile\b', code)))
    has_func    = int(bool(re.search(
        r'\bdef\b|\bfunction\b|\bfunc\b|\bvoid\b|\bint\b\s+\w+\s*\(', code)))

    prefix = (f"[TYPE:{content_type} LEN:{complexity} "
              f"CLASS:{has_class} LOOP:{has_loop} FUNC:{has_func}]")
    return f"{prefix} {code}"

for df in [train_df, val_df, test_df]:
    df["code"] = [classify_and_prefix(c) if c else "" for c in df["code"]]

print("Sample prefix:", train_df["code"].iloc[0][:100])

# Verify type distribution on test set
test_types = pd.Series(test_df["code"]).str.extract(r'\[TYPE:(\w+)')[0]
print("\nTest content type distribution:")
print(test_types.value_counts())


Sample prefix: [TYPE:clean LEN:short CLASS:0 LOOP:1 FUNC:0] (a, b, c, d) = [int(x) for x in input().split()]
k = in

Test content type distribution:
0
clean       479749
mixed        12581
fragment      7670
Name: count, dtype: int64


In [10]:
# Delexicalization
# Replaces identifiers/strings/numbers with generic tokens.
# Used as a second "view" during training (KL loss) and at
# test time for ensembling. Forces model to rely on structural
# patterns that generalize across domains and languages.

_re_block = re.compile(r"/\*.*?\*/", re.S)
_re_line  = re.compile(r"//.*?$|#.*?$", re.M)
_re_str   = re.compile(r"\"(?:\\.|[^\"\\])*\"|'(?:\\.|[^'\\])*'", re.S)
_re_num   = re.compile(r"\b\d+(\.\d+)?\b")
_re_ident = re.compile(r"\b[A-Za-z_][A-Za-z0-9_]*\b")

def delexicalize(code: str) -> str:
    if not code:
        return ""
    s = _re_block.sub(" ", code)
    s = _re_line.sub(" ", s)
    s = _re_str.sub(" STR ", s)
    s = _re_num.sub(" NUM ", s)
    s = _re_ident.sub(" ID ", s)
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()

In [11]:
# Mixed Content Augmentation
# Simulates mixed/fragment samples during training by
# injecting generic text into clean code. This is augmentation
# of existing samples

FILLER_TEXTS = [
    "function description",
    "helper method",
    "main logic",
    "initialization",
    "return value",
    "loop body",
    "base case",
    "edge case",
    "input handling",
    "output format",
]

def mix_code_with_text(code: str, mix_prob: float = 0.4) -> str:
    if random.random() > mix_prob or not code:
        return code
    lines = code.splitlines()
    if len(lines) < 3:
        return code
    n_inserts = random.randint(1, 3)
    for _ in range(n_inserts):
        pos  = random.randint(0, len(lines))
        text = random.choice(FILLER_TEXTS)
        lines.insert(pos, f"# {text}")
    return "\n".join(lines)

In [12]:
# Token Dropout Augmentation
# Randomly masks TOKEN_DROPOUT_RATE % of tokens during training.
# Forces the model not to rely on specific tokens (language
# keywords, library names) that only appear in seen languages.
# This simulates seeing "incomplete" code from unknown languages.

TOKEN_DROPOUT_RATE = 0.15  # mask 15% of tokens

def apply_token_dropout(input_ids: torch.Tensor,
                         mask_token_id: int,
                         pad_token_id: int,
                         rate: float = TOKEN_DROPOUT_RATE) -> torch.Tensor:
    """Randomly replace rate% of non-special tokens with [MASK]."""
    result = input_ids.clone()
    # Don't mask CLS, SEP, PAD tokens (ids 0, 1, 2 typically)
    special_mask = (result <= 3) | (result == pad_token_id)
    rand = torch.rand_like(result, dtype=torch.float)
    drop_mask = (rand < rate) & ~special_mask
    result[drop_mask] = mask_token_id
    return result

In [13]:
#  Model + Tokenizer
# UniXcoder: trained on CodeSearchNet across 6 languages
# (Ruby, JS, Go, Python, Java, PHP) → best cross-lingual transfer.
# Better than GraphCodeBERT for unseen language generalization.

MODEL_NAME = "microsoft/unixcoder-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

HALF_LEN   = 256   # first 256 + last 256 = 512 effective tokens
MAX_LENGTH = HALF_LEN * 2

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [14]:
#  First+Last Tokenization
# Captures both start (imports/signatures) and end (main logic)
# of long files instead of just truncating at MAX_LENGTH tokens.

def first_last_encode(code: str) -> Dict[str, List[int]]:
    tokens = tokenizer.encode(
        code, truncation=False, add_special_tokens=False
    )
    if len(tokens) <= MAX_LENGTH - 2:
        enc = tokenizer(code, truncation=True, max_length=MAX_LENGTH, padding=False)
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

    half     = HALF_LEN - 1
    combined = tokens[:half] + tokens[-half:]
    cls_id   = tokenizer.cls_token_id
    sep_id   = tokenizer.sep_token_id
    input_ids      = [cls_id] + combined + [sep_id]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask}

def tokenize_with_aug(batch):
    """Tokenize original + delex for training/validation."""
    results = {
        "input_ids": [], "attention_mask": [],
        "input_ids_aug": [], "attention_mask_aug": [],
        "input_ids_mix": [], "attention_mask_mix": [],
    }
    for code in batch["code"]:
        code  = code if code else ""
        delex = delexicalize(code)
        mixed = mix_code_with_text(code, mix_prob=0.4)
        enc_main = first_last_encode(code)
        enc_aug  = first_last_encode(delex)
        enc_mix  = first_last_encode(mixed)
        results["input_ids"].append(enc_main["input_ids"])
        results["attention_mask"].append(enc_main["attention_mask"])
        results["input_ids_aug"].append(enc_aug["input_ids"])
        results["attention_mask_aug"].append(enc_aug["attention_mask"])
        results["input_ids_mix"].append(enc_mix["input_ids"])
        results["attention_mask_mix"].append(enc_mix["attention_mask"])
    return results

def tokenize_test_fn(batch):
    """Tokenize original + delex for test-time ensembling."""
    results = {
        "input_ids": [], "attention_mask": [],
        "input_ids_delex": [], "attention_mask_delex": [],
    }
    for code in batch["code"]:
        code  = code if code else ""
        delex = delexicalize(code)
        enc_main  = first_last_encode(code)
        enc_delex = first_last_encode(delex)
        results["input_ids"].append(enc_main["input_ids"])
        results["attention_mask"].append(enc_main["attention_mask"])
        results["input_ids_delex"].append(enc_delex["input_ids"])
        results["attention_mask_delex"].append(enc_delex["attention_mask"])
    return results

# Check token lengths on a sample
sample_lengths = [
    len(tokenizer.encode(c, truncation=False, add_special_tokens=False))
    for c in train_df["code"].sample(min(2000, len(train_df)), random_state=SEED)
]
print(f"\nToken length distribution:")
print(f"  p50={np.percentile(sample_lengths,50):.0f}  "
      f"p75={np.percentile(sample_lengths,75):.0f}  "
      f"p90={np.percentile(sample_lengths,90):.0f}  "
      f"p99={np.percentile(sample_lengths,99):.0f}  "
      f"max={max(sample_lengths)}")

# Build datasets
ds = DatasetDict({
    "train":      Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df,   preserve_index=False),
    "test":       Dataset.from_pandas(test_df,  preserve_index=False),
})

# Tokenize — num_proc=1 for Colab stability
remove_train = [c for c in ds["train"].column_names      if c not in ["ID","code","label"]]
remove_val   = [c for c in ds["validation"].column_names if c not in ["ID","code","label"]]
remove_test  = [c for c in ds["test"].column_names       if c not in ["ID","code"]]

print("\nTokenizing train...")
train_tok = ds["train"].map(tokenize_with_aug, batched=True, batch_size=256,
                             remove_columns=remove_train, num_proc=1)
train_tok = train_tok.rename_column("label", "labels")

print("Tokenizing validation...")
val_tok = ds["validation"].map(tokenize_with_aug, batched=True, batch_size=256,
                                remove_columns=remove_val, num_proc=1)
val_tok = val_tok.rename_column("label", "labels")

print("Tokenizing test...")
test_tok = ds["test"].map(tokenize_test_fn, batched=True, batch_size=256,
                           remove_columns=remove_test, num_proc=1)

tokenized_ds = DatasetDict({
    "train":      train_tok,
    "validation": val_tok,
    "test":       test_tok,
})
print("\nTokenized datasets:")
print(tokenized_ds)


Token length distribution:
  p50=214  p75=390  p90=794  p99=1359  max=23691

Tokenizing train...


Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

Tokenizing validation...


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Tokenizing test...


Map:   0%|          | 0/500000 [00:00<?, ? examples/s]


Tokenized datasets:
DatasetDict({
    train: Dataset({
        features: ['code', 'labels', 'input_ids', 'attention_mask', 'input_ids_aug', 'attention_mask_aug', 'input_ids_mix', 'attention_mask_mix'],
        num_rows: 500000
    })
    validation: Dataset({
        features: ['code', 'labels', 'input_ids', 'attention_mask', 'input_ids_aug', 'attention_mask_aug', 'input_ids_mix', 'attention_mask_mix'],
        num_rows: 100000
    })
    test: Dataset({
        features: ['ID', 'code', 'input_ids', 'attention_mask', 'input_ids_delex', 'attention_mask_delex'],
        num_rows: 500000
    })
})


In [15]:
# Collator
@dataclass
class DualViewCollator:
    """
    Pads both original and aug views for training/eval.
    Falls back gracefully when aug columns are absent (test).
    """
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        has_aug = "input_ids_aug" in features[0]
        has_mix = "input_ids_mix" in features[0]
        main, aug, mix = [], [], []
        for f in features:
            main.append({
                "input_ids":      f["input_ids"],
                "attention_mask": f["attention_mask"],
                **( {"labels": f["labels"]} if "labels" in f else {} ),
            })
            if has_aug:
                aug.append({
                    "input_ids":      f["input_ids_aug"],
                    "attention_mask": f["attention_mask_aug"],
                })
            if has_mix:
                mix.append({
                    "input_ids":      f["input_ids_mix"],
                    "attention_mask": f["attention_mask_mix"],
                })
        batch = self.tokenizer.pad(main, return_tensors="pt")
        if has_aug:
            ab = self.tokenizer.pad(aug, return_tensors="pt")
            batch["input_ids_aug"]      = ab["input_ids"]
            batch["attention_mask_aug"] = ab["attention_mask"]
        if has_mix:
            mb = self.tokenizer.pad(mix, return_tensors="pt")
            batch["input_ids_mix"]      = mb["input_ids"]
            batch["attention_mask_mix"] = mb["attention_mask"]
        return batch

data_collator = DualViewCollator(tokenizer=tokenizer)


In [16]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:

f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "acc":      acc_metric.compute(predictions=preds, references=labels)["accuracy"],
    }

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

In [18]:
# ConsistencyTrainer (CE + KL)
class ConsistencyTrainer(Trainer):
    """
    Fused forward pass over [original; delex] concatenated batch.
    CE loss on original logits + KL divergence between original
    and delex distributions.

    KL rather than CE on aug: we don't assert aug labels are
    correct (delexicalization may slightly change semantics).
    We only want prediction CONSISTENCY between views.
    """
    def __init__(self, *args, lambda_kl: float = 0.3, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_kl = lambda_kl
        self.mask_token_id  = tokenizer.mask_token_id or tokenizer.unk_token_id
        self.pad_token_id   = tokenizer.pad_token_id

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels             = inputs.pop("labels",             None)
        input_ids_aug      = inputs.pop("input_ids_aug",      None)
        attention_mask_aug = inputs.pop("attention_mask_aug", None)
        input_ids_mix      = inputs.pop("input_ids_mix",      None)
        attention_mask_mix = inputs.pop("attention_mask_mix", None)

        # Eval / test path — plain forward pass
        if input_ids_aug is None or labels is None:
            logits = model(**inputs).logits
            loss   = F.cross_entropy(logits, labels) if labels is not None else None
            out = SequenceClassifierOutput(loss=loss, logits=logits)
            return (loss, out) if return_outputs else out.loss

        # Apply token dropout to original view during training
        dropped_ids = apply_token_dropout(
            inputs["input_ids"],
            mask_token_id=self.mask_token_id,
            pad_token_id=self.pad_token_id,
        )

        # Pad all views to the same length
        def pad_to(tensor, length, val):
            if tensor.shape[1] < length:
                tensor = F.pad(tensor, (0, length - tensor.shape[1]), value=val)
            return tensor

        lengths = [dropped_ids.shape[1], input_ids_aug.shape[1]]
        if input_ids_mix is not None:
            lengths.append(input_ids_mix.shape[1])
        max_len = max(lengths)

        dropped_ids        = pad_to(dropped_ids,               max_len, self.pad_token_id)
        input_ids_aug      = pad_to(input_ids_aug,             max_len, self.pad_token_id)
        attn_main          = pad_to(inputs["attention_mask"],  max_len, 0)
        attention_mask_aug = pad_to(attention_mask_aug,        max_len, 0)

        all_ids   = [dropped_ids, input_ids_aug]
        all_masks = [attn_main,   attention_mask_aug]

        if input_ids_mix is not None:
            input_ids_mix      = pad_to(input_ids_mix,      max_len, self.pad_token_id)
            attention_mask_mix = pad_to(attention_mask_mix, max_len, 0)
            all_ids.append(input_ids_mix)
            all_masks.append(attention_mask_mix)

        # Single fused forward pass
        combined_out = model(
            input_ids=torch.cat(all_ids,   dim=0),
            attention_mask=torch.cat(all_masks, dim=0),
        )

        bsz        = dropped_ids.size(0)
        logits     = combined_out.logits[:bsz]
        logits_aug = combined_out.logits[bsz:2*bsz]

        # CE with label smoothing
        loss_ce = F.cross_entropy(logits, labels, label_smoothing=0.1)
        # Symmetric KL: original ↔ delex
        p = F.log_softmax(logits,     dim=-1)
        q = F.log_softmax(logits_aug, dim=-1)
        loss_kl_delex = (
            F.kl_div(p, q.detach().exp(), reduction="batchmean") +
            F.kl_div(q, p.detach().exp(), reduction="batchmean")
        ) / 2.0

        loss = loss_ce + self.lambda_kl * loss_kl_delex

        # Symmetric KL: original ↔ mixed (softer weight)
        if input_ids_mix is not None:
            logits_mix = combined_out.logits[2*bsz:]
            r = F.log_softmax(logits_mix, dim=-1)
            loss_kl_mix = (
                F.kl_div(p, r.detach().exp(), reduction="batchmean") +
                F.kl_div(r, p.detach().exp(), reduction="batchmean")
            ) / 2.0
            loss = loss + (self.lambda_kl * 0.5) * loss_kl_mix

        out = SequenceClassifierOutput(loss=loss_ce, logits=logits)
        return (loss, out) if return_outputs else loss



In [19]:
#Training args
# Label smoothing (label_smoothing_factor=0.1)
# Prevents overfit to generator-specific artifacts in train set,
# improving robustness to unseen generators at evaluation time.

training_args = TrainingArguments(
    output_dir                = CHECKPOINT_DIR,
    remove_unused_columns     = False,
    learning_rate             = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    gradient_accumulation_steps = 4,
    num_train_epochs          = 1,
    weight_decay              = 0.01,
    warmup_steps              = 470,
    fp16                      = True,
    dataloader_num_workers    = 2,
    eval_strategy             = "steps",
    eval_steps                = 1000,
    save_steps                = 1000,
    save_total_limit          = 2,
    logging_steps             = 200,
    load_best_model_at_end    = True,
    metric_for_best_model     = "macro_f1",
    greater_is_better         = True,
    report_to                 = "none",
    seed                      = SEED,
)

trainer = ConsistencyTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized_ds["train"],
    eval_dataset     = tokenized_ds["validation"],
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
    lambda_kl        = 0.5,
)

In [ ]:
# Train
train_result = trainer.train(resume_from_checkpoint=True)
print("Train metrics:", train_result.metrics)

# Save model
trainer.save_model(os.path.join(CHECKPOINT_DIR, "best_model"))
tokenizer.save_pretrained(os.path.join(CHECKPOINT_DIR, "best_model"))
print("Model saved to Drive!")

eval_result = trainer.evaluate()
print("Validation:", eval_result)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Step,Training Loss,Validation Loss,Macro F1,Acc
5000,0.895054,0.222543,0.995291,0.995300
6000,0.888278,0.222784,0.995741,0.995750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Macro F1,Acc
5000,0.895054,0.222543,0.995291,0.995300
6000,0.888278,0.222784,0.995741,0.995750


In [ ]:
print("Tokenizing mixed test...")

def tokenize_mixed_test(batch):
    results = {"input_ids": [], "attention_mask": []}
    for code in batch["code"]:
        mixed = mix_code_with_text(code, mix_prob=1.0)
        enc   = first_last_encode(mixed)
        results["input_ids"].append(enc["input_ids"])
        results["attention_mask"].append(enc["attention_mask"])
    return results

test_mixed_df  = test_df.copy()
test_mixed_ds  = Dataset.from_pandas(test_mixed_df, preserve_index=False)
remove_cols    = [c for c in test_mixed_ds.column_names if c not in ["ID", "code"]]
test_mixed_tok = test_mixed_ds.map(
    tokenize_mixed_test, batched=True, batch_size=64,
    remove_columns=remove_cols, num_proc=1
)

print("Predicting on test (original)...")
orig_logits = trainer.predict(tokenized_ds["test"]).predictions

print("Predicting on test (delex)...")
test_delex_ds = (
    tokenized_ds["test"]
    .remove_columns([
        c for c in tokenized_ds["test"].column_names
        if c not in ["input_ids_delex", "attention_mask_delex"]
    ])
    .rename_column("input_ids_delex",      "input_ids")
    .rename_column("attention_mask_delex", "attention_mask")
)
delex_logits = trainer.predict(test_delex_ds).predictions

print("Predicting on test (mixed)...")
mixed_logits = trainer.predict(test_mixed_tok).predictions

# 3-way ensemble: equal weight average
ensemble_logits = (orig_logits + delex_logits + mixed_logits) / 3.0
test_preds      = np.argmax(ensemble_logits, axis=-1).astype(int)

print(f"\nTest prediction distribution:")
print(pd.Series(test_preds).value_counts(normalize=True).mul(100).round(1))

# Save submission
submission = pd.DataFrame({
    "ID":    tokenized_ds["test"]["ID"],
    "label": test_preds,
})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv(os.path.join(CHECKPOINT_DIR, "submission.csv"), index=False)

from google.colab import files
files.download("/content/submission.csv")
print(submission.head())
print("Saved submission.csv!")

In [ ]:
print(submission["label"].value_counts())
print(submission["label"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# Validation report + confusion matrix
val_pred   = trainer.predict(tokenized_ds["validation"])
val_logits = val_pred.predictions
val_labels = val_pred.label_ids
val_preds  = np.argmax(val_logits, axis=-1)

print(classification_report(val_labels, val_preds, digits=4))

for normalize, fmt, title in [
    (None,   "d",    "Confusion Matrix (Validation)"),
    ("true", ".2f",  "Confusion Matrix (Validation, normalized)"),
]:
    cm   = confusion_matrix(val_labels, val_preds, labels=[0, 1], normalize=normalize)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Human(0)", "Machine(1)"])
    plt.figure(figsize=(5, 5))
    disp.plot(values_format=fmt)
    plt.title(title)
    plt.tight_layout()
    plt.show()